# RAG From Scratch — Retrieval Engineering Extension

> **Purpose**: This notebook fills the retrieval engineering gaps in the original `rag-from-scratch` series.  
> Each section is labeled with where it fits in the original notebook sequence.

| Section | Topic | Insert After |
|---------|-------|-------------|
| Part A | BM25 / Sparse Retrieval | Notebook 1-4, after Part 3 (Retrieval) |
| Part B | Hybrid Search with RRF | Notebook 1-4, after Part A above |
| Part C | Embedding Model Selection | Notebook 1-4, after Part 1 (Indexing) |
| Part D | Cross-Encoder Reranking | Notebook 12-14, after Part 13 (RAPTOR) |
| Part E | Full Pipeline Integration | Standalone — combines everything |

---

## Environment Setup

```bash
pip install langchain langchain-community langchain-openai langchain-chroma
pip install rank_bm25 sentence-transformers FlagEmbedding
pip install tiktoken chromadb
```

In [ ]:
import os
# os.environ["OPENAI_API_KEY"] = "your-key-here"

# Shared imports used throughout this notebook
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

## Telecom Domain Test Data

We use 5GC (5G Core) technical documents as our test corpus.  
This makes the exercises directly relevant to domain-specific RAG challenges:
- Heavy use of abbreviations (AMF, SMF, UPF, PFCP, N2, N4)
- Mixed Chinese-English content
- Precise terminology where keyword matching matters

In [ ]:
# 5GC domain documents for testing
docs = [
    Document(
        page_content="""The AMF (Access and Mobility Management Function) is responsible for 
        registration management, connection management, reachability management, and mobility 
        management. It terminates the N1 and N2 interfaces from the UE and gNB respectively. 
        The AMF selects the appropriate SMF for PDU session establishment.""",
        metadata={"source": "TS23.501", "clause": "6.2.1", "nf": "AMF", "release": "R16"}
    ),
    Document(
        page_content="""The SMF (Session Management Function) is responsible for session 
        management including session establishment, modification, and release. It allocates 
        IP addresses to UEs and selects the appropriate UPF. The SMF communicates with the 
        AMF via the N11 interface and controls the UPF via the N4 interface using PFCP 
        (Packet Forwarding Control Protocol).""",
        metadata={"source": "TS23.501", "clause": "6.2.2", "nf": "SMF", "release": "R16"}
    ),
    Document(
        page_content="""The UPF (User Plane Function) handles packet routing and forwarding, 
        QoS enforcement, and traffic measurement. It serves as the anchor point for intra/inter-RAT 
        mobility and the external PDU session point of interconnect to the data network. 
        The UPF is controlled by the SMF through PFCP sessions over the N4 interface.""",
        metadata={"source": "TS23.501", "clause": "6.2.3", "nf": "UPF", "release": "R16"}
    ),
    Document(
        page_content="""PFCP Session Establishment procedure: The SMF sends a PFCP Session 
        Establishment Request to the UPF containing PDR (Packet Detection Rules), FAR 
        (Forwarding Action Rules), QER (QoS Enforcement Rules), and URR (Usage Reporting 
        Rules). The UPF responds with a PFCP Session Establishment Response including the 
        F-SEID (Fully qualified SEID) for subsequent modifications.""",
        metadata={"source": "TS29.244", "clause": "7.2.2", "nf": "UPF", "release": "R16"}
    ),
    Document(
        page_content="""In Release 17, the AMF is enhanced with support for Non-3GPP access 
        via N3IWF and TNGF. The N2 interface procedures are extended to support multiple 
        PDU sessions per UE context. Edge computing support is added through LADN 
        (Local Area Data Network) configuration and DNAI (DN Access Identifier) management.""",
        metadata={"source": "TS23.501", "clause": "6.2.1.1", "nf": "AMF", "release": "R17"}
    ),
    Document(
        page_content="""Network Slice selection is performed by the NSSF (Network Slice 
        Selection Function). The AMF queries the NSSF with Requested NSSAI and Subscribed 
        NSSAI to determine the Allowed NSSAI and target AMF set. S-NSSAI consists of SST 
        (Slice/Service Type) and optional SD (Slice Differentiator).""",
        metadata={"source": "TS23.501", "clause": "5.15", "nf": "NSSF", "release": "R16"}
    ),
    Document(
        page_content="""The NRF (Network Repository Function) supports NF service discovery 
        and NF profile management. Other NFs register with the NRF via Nnrf_NFManagement 
        service and discover available NF instances via Nnrf_NFDiscovery service. 
        The NRF maintains NF profiles containing FQDN, IP addresses, supported services, 
        and capacity information.""",
        metadata={"source": "TS23.501", "clause": "6.2.6", "nf": "NRF", "release": "R16"}
    ),
    Document(
        page_content="""PDU Session Establishment call flow: 1) UE sends NAS PDU Session 
        Establishment Request to AMF. 2) AMF selects SMF based on DNN, S-NSSAI, and 
        subscription data. 3) SMF creates SM context via Nsmf_PDUSession_CreateSMContext. 
        4) SMF selects UPF and sends PFCP Session Establishment Request. 5) N2 PDU Session 
        Resource Setup Request is sent to gNB via AMF.""",
        metadata={"source": "TS23.502", "clause": "4.3.2.2", "nf": "SMF", "release": "R16"}
    ),
]

print(f"Loaded {len(docs)} telecom domain documents")
for d in docs:
    print(f"  [{d.metadata['source']} {d.metadata['clause']}] {d.metadata['nf']} - {d.page_content[:60]}...")

---

# Part A: BM25 / Sparse Retrieval

> 📍 **Insert after**: Notebook 1-4, Part 3 (Retrieval)  
> **Why this matters**: The original notebook only covers dense vector search. BM25 is critical 
> for domain-specific RAG where exact terminology matching is essential.

## The Problem with Pure Dense Retrieval

Dense embeddings map text into a semantic space where similar *meanings* are close together.  
But in telecom documents, **precision often matters more than semantic similarity**:

| Query | Desired Match | Dense Search Risk |
|-------|--------------|-------------------|
| "N4 interface" | Docs mentioning N4 specifically | Returns any interface-related content |
| "PFCP session" | PFCP protocol details | Returns generic session management docs |
| "AMF R17 changes" | R17-specific AMF content | Mixes R16 and R17 content |

BM25 solves this by doing **exact token matching** with TF-IDF weighting.

In [ ]:
# ── Part A1: Basic BM25 Retriever ──

from langchain_community.retrievers import BM25Retriever

# Create BM25 retriever from our documents
bm25_retriever = BM25Retriever.from_documents(docs, k=3)

# Test 1: Query with specific protocol abbreviation
query1 = "PFCP session establishment"
bm25_results = bm25_retriever.invoke(query1)

print(f"Query: '{query1}'")
print(f"Results ({len(bm25_results)}):")
for i, doc in enumerate(bm25_results):
    print(f"  [{i+1}] [{doc.metadata['source']} {doc.metadata['clause']}] "
          f"{doc.page_content[:80]}...")
print()

In [ ]:
# ── Part A2: Compare BM25 vs Dense on abbreviation-heavy queries ──

# Create dense retriever
embeddings = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(docs, embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Test queries that highlight BM25 strengths
test_queries = [
    "N4 interface PFCP",           # Protocol-specific
    "NSSF slice selection S-NSSAI", # Abbreviation-heavy
    "how does session management work",  # Semantic / conceptual
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: '{query}'")
    
    bm25_results = bm25_retriever.invoke(query)
    dense_results = dense_retriever.invoke(query)
    
    print(f"\n  BM25 Top-1: [{bm25_results[0].metadata['source']} "
          f"{bm25_results[0].metadata['clause']}] {bm25_results[0].metadata['nf']}")
    print(f"  Dense Top-1: [{dense_results[0].metadata['source']} "
          f"{dense_results[0].metadata['clause']}] {dense_results[0].metadata['nf']}")

### Key Takeaway

- **BM25 wins** on abbreviation-heavy, protocol-specific queries (N4, PFCP, S-NSSAI)
- **Dense wins** on conceptual queries ("how does session management work")
- **Neither alone is sufficient** → This motivates hybrid search (Part B)

---

# Part B: Hybrid Search with Reciprocal Rank Fusion (RRF)

> 📍 **Insert after**: Part A above (still in Notebook 1-4 flow)  
> **Key concept**: Combine BM25 (exact matching) + Dense (semantic matching) using RRF.

## Why RRF over Linear Score Combination?

Two retrievers produce scores on completely different scales:
- BM25: scores range from 0 to ~25 (based on TF-IDF)
- Dense cosine similarity: scores range from 0 to 1

**Linear combination** (`0.4 * bm25_score + 0.6 * dense_score`) fails because 
the scales are incomparable — a BM25 score of 15 and a cosine similarity of 0.85 
can't be meaningfully added.

**RRF** uses only **rank positions**, not raw scores:

$$\text{RRF}(d) = \sum_{r \in \text{retrievers}} \frac{1}{k + \text{rank}_r(d)}$$

where $k = 60$ is the standard constant. This makes it scale-invariant.

In [ ]:
# ── Part B1: LangChain EnsembleRetriever (built-in RRF) ──

from langchain.retrievers import EnsembleRetriever

# Create hybrid retriever with BM25 (40%) + Dense (60%)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.4, 0.6]  # Weights applied to RRF scores
)

# Test the same queries
for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: '{query}'")
    
    hybrid_results = ensemble_retriever.invoke(query)
    
    print(f"  Hybrid Results:")
    for i, doc in enumerate(hybrid_results[:3]):
        print(f"    [{i+1}] [{doc.metadata['source']} {doc.metadata['clause']}] "
              f"{doc.metadata['nf']} - {doc.page_content[:60]}...")

In [ ]:
# ── Part B2: Manual RRF Implementation (for understanding) ──

def reciprocal_rank_fusion(
    results_list: list[list[Document]], 
    k: int = 60
) -> list[tuple[Document, float]]:
    """
    RRF: score(d) = Σ 1/(k + rank_i)
    
    k=60 is the standard constant from Cormack et al., 2009.
    Using rank-based fusion means we don't need to normalize scores
    across different retrievers.
    """
    doc_scores = {}  # doc_id -> (doc, score)
    
    for results in results_list:
        for rank, doc in enumerate(results):
            doc_id = f"{doc.metadata['source']}_{doc.metadata['clause']}"
            if doc_id not in doc_scores:
                doc_scores[doc_id] = (doc, 0.0)
            current_doc, current_score = doc_scores[doc_id]
            doc_scores[doc_id] = (current_doc, current_score + 1.0 / (k + rank + 1))
    
    # Sort by fused score, descending
    sorted_docs = sorted(doc_scores.values(), key=lambda x: x[1], reverse=True)
    return sorted_docs

# Demo: manual RRF on a query
query = "PFCP session establishment N4"
bm25_results = bm25_retriever.invoke(query)
dense_results = dense_retriever.invoke(query)

fused = reciprocal_rank_fusion([bm25_results, dense_results])

print(f"Query: '{query}'\n")
print("Manual RRF results:")
for doc, score in fused[:5]:
    print(f"  Score: {score:.4f} | [{doc.metadata['source']} {doc.metadata['clause']}] "
          f"{doc.metadata['nf']}") 

In [ ]:
# ── Part B3: Weight Tuning Experiment ──
# How do different BM25/Dense weight ratios affect results?

query = "N4 interface PFCP"
print(f"Query: '{query}'\n")

weight_configs = [
    (0.0, 1.0, "Pure Dense"),
    (0.3, 0.7, "Dense-heavy"),
    (0.5, 0.5, "Balanced"),
    (0.7, 0.3, "BM25-heavy"),
    (1.0, 0.0, "Pure BM25"),
]

for w_bm25, w_dense, label in weight_configs:
    retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, dense_retriever],
        weights=[w_bm25, w_dense]
    )
    results = retriever.invoke(query)
    top1 = results[0]
    print(f"  {label:15s} ({w_bm25:.1f}/{w_dense:.1f}): "
          f"Top-1 = [{top1.metadata['source']} {top1.metadata['clause']}] {top1.metadata['nf']}")

### Key Takeaway

- For **telecom domain** with heavy abbreviation use, BM25 weight of 0.4-0.5 works well
- The "right" weight depends on your query distribution — evaluate on your own test set
- RRF is robust because it only uses ranks, not raw scores

---

# Part C: Embedding Model Selection

> 📍 **Insert after**: Notebook 1-4, Part 1 (Indexing), during embedding setup  
> **Key insight**: The choice of embedding model significantly impacts retrieval quality, 
> especially for domain-specific and multilingual content.

## Selection Criteria for 5GC Documents

| Criterion | Why It Matters |
|-----------|---------------|
| Chinese-English mixed support | 5GC docs are often Chinese with embedded English terms |
| Dimension size | Affects storage cost and retrieval speed |
| Domain terminology handling | "AMF", "N4", "PFCP" must be meaningful embeddings |
| Deployment flexibility | Local deployment vs. API dependency |

In [ ]:
# ── Part C1: Embedding Model Comparison Framework ──

from langchain_openai import OpenAIEmbeddings
import numpy as np

# Model configurations to compare
# In practice, you'd also test BGE-M3 and BGE-large-en locally
# Here we demonstrate the comparison methodology with OpenAI variants

models = {
    "text-embedding-3-small": OpenAIEmbeddings(model="text-embedding-3-small"),
    "text-embedding-3-large": OpenAIEmbeddings(model="text-embedding-3-large"),
}

# Test queries: mix of abbreviation-heavy and conceptual
eval_pairs = [
    # (query, expected_top_doc_clause) - manually labeled
    ("PFCP session establishment procedure", "7.2.2"),     # exact match
    ("N4 interface between SMF and UPF", "6.2.2"),          # should find SMF doc
    ("how does the network select a slice", "5.15"),        # conceptual
    ("AMF changes in Release 17", "6.2.1.1"),              # version-specific
    ("PDU session setup call flow", "4.3.2.2"),             # procedural
]

print("Embedding Model Comparison")
print("=" * 60)

for model_name, embedding_model in models.items():
    # Build vectorstore with this model
    vs = Chroma.from_documents(docs, embedding_model, 
                                collection_name=model_name.replace("-", "_"))
    retriever = vs.as_retriever(search_kwargs={"k": 1})
    
    correct = 0
    for query, expected_clause in eval_pairs:
        result = retriever.invoke(query)
        if result and result[0].metadata["clause"] == expected_clause:
            correct += 1
    
    accuracy = correct / len(eval_pairs)
    print(f"\n  {model_name}:")
    print(f"    Hit@1 = {correct}/{len(eval_pairs)} ({accuracy:.0%})")

### Recommendation for 5GC RAG

For a production 5GC RAG system, **BGE-M3** is the recommended embedding model:

```python
# BGE-M3 (requires local GPU or CPU inference)
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
embeddings = model.encode(
    texts,
    return_dense=True,    # 768d dense vector
    return_sparse=True,   # learned sparse (can replace BM25)
)
```

**Why BGE-M3 over OpenAI embeddings:**
1. **Native Chinese-English support** — trained on multilingual data
2. **768d** — good balance of quality and storage efficiency (vs. 1536d/3072d for OpenAI)
3. **Local deployment** — no API cost, no data leaving your network
4. **Built-in sparse output** — can optionally replace BM25 with learned sparse retrieval

---

# Part D: Cross-Encoder Reranking

> 📍 **Insert after**: Notebook 12-14, Part 13 (RAPTOR)  
> **Key concept**: Bi-encoders (BGE, OpenAI) are fast but less precise. Cross-encoders 
> are slow but much more accurate. Use bi-encoders for coarse retrieval (top-20), 
> then cross-encoders for fine reranking (top-5).

## Bi-Encoder vs Cross-Encoder

```
Bi-Encoder (fast, used for initial retrieval):
  Query  ──→ [Encoder] ──→ query_vector  ─┐
                                            ├── cosine similarity
  Doc    ──→ [Encoder] ──→ doc_vector    ─┘

Cross-Encoder (slow, used for reranking):
  [Query + Doc] ──→ [Full Transformer] ──→ relevance_score
```

The cross-encoder sees query and document **together**, enabling attention across 
both — it understands "N4 interface" in the query should match "controlled via the 
N4 interface using PFCP" in the document, even if the bi-encoder missed the connection.

In [ ]:
# ── Part D1: Cross-Encoder Reranking with sentence-transformers ──

from sentence_transformers import CrossEncoder

# Load cross-encoder model
# bge-reranker-v2-m3 is recommended for multilingual; 
# ms-marco-MiniLM is a lightweight alternative
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_with_cross_encoder(
    query: str, 
    candidates: list[Document], 
    top_k: int = 3
) -> list[tuple[Document, float]]:
    """Rerank candidate documents using a cross-encoder."""
    # Create query-document pairs
    pairs = [[query, doc.page_content] for doc in candidates]
    
    # Score each pair
    scores = cross_encoder.predict(pairs)
    
    # Sort by score, return top_k
    scored_docs = sorted(
        zip(candidates, scores), 
        key=lambda x: x[1], 
        reverse=True
    )
    return scored_docs[:top_k]

# Demo: rerank the hybrid retrieval results
query = "How does SMF control UPF for packet forwarding?"
print(f"Query: '{query}'\n")

# Step 1: Coarse retrieval (top-5 from hybrid)
coarse_results = ensemble_retriever.invoke(query)[:5]
print("Before reranking (hybrid retrieval order):")
for i, doc in enumerate(coarse_results):
    print(f"  [{i+1}] [{doc.metadata['source']} {doc.metadata['clause']}] "
          f"{doc.metadata['nf']}")

# Step 2: Fine reranking (cross-encoder)
reranked = rerank_with_cross_encoder(query, coarse_results, top_k=3)
print(f"\nAfter cross-encoder reranking:")
for i, (doc, score) in enumerate(reranked):
    print(f"  [{i+1}] Score: {score:.4f} | [{doc.metadata['source']} "
          f"{doc.metadata['clause']}] {doc.metadata['nf']}")

In [ ]:
# ── Part D2: Full Retrieval Pipeline with Reranking ──

def full_retrieval_pipeline(
    query: str,
    bm25_retriever,
    dense_retriever,
    cross_encoder,
    bm25_weight: float = 0.4,
    coarse_k: int = 10,
    final_k: int = 3,
) -> list[Document]:
    """
    Complete retrieval pipeline:
    1. BM25 + Dense hybrid retrieval (coarse, fast)
    2. RRF fusion
    3. Cross-encoder reranking (fine, slow)
    4. Return top-k reranked documents
    """
    # Step 1: Hybrid retrieval
    hybrid = EnsembleRetriever(
        retrievers=[bm25_retriever, dense_retriever],
        weights=[bm25_weight, 1 - bm25_weight]
    )
    candidates = hybrid.invoke(query)[:coarse_k]
    
    # Step 2: Cross-encoder reranking
    pairs = [[query, doc.page_content] for doc in candidates]
    scores = cross_encoder.predict(pairs)
    
    scored = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in scored[:final_k]]

# Test the full pipeline
queries = [
    "PFCP session establishment on N4 interface",
    "What changed in AMF for Release 17?",
    "How is network slicing configured?",
]

for query in queries:
    print(f"\nQuery: '{query}'")
    results = full_retrieval_pipeline(
        query, bm25_retriever, dense_retriever, cross_encoder
    )
    for i, doc in enumerate(results):
        print(f"  [{i+1}] [{doc.metadata['source']} {doc.metadata['clause']}] "
              f"{doc.metadata['nf']}")

### Performance Characteristics

| Stage | Method | Speed | Accuracy | Candidate Size |
|-------|--------|-------|----------|---------------|
| Coarse | BM25 + Dense (RRF) | ~10ms | Medium | full corpus → top-20 |
| Fine | Cross-Encoder | ~100ms | High | top-20 → top-5 |

**Why not use cross-encoder for everything?**  
Cross-encoders require encoding query+doc together, so they scale as O(N) with corpus size.  
On 100K documents, that's ~100K forward passes per query — way too slow.  
Bi-encoders pre-compute doc embeddings, so retrieval is a fast nearest-neighbor search.

---

# Part E: Full RAG Pipeline Integration

> 📍 **Standalone section** — combines all retrieval improvements with generation  
> This is the complete pipeline: Query → Hybrid Retrieval → Reranking → Generation with Citations

In [ ]:
# ── Part E1: End-to-End RAG with Hybrid Retrieval + Reranking ──

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# RAG prompt with citation requirement
rag_prompt = ChatPromptTemplate.from_template(
"""You are a 5G Core network expert. Answer the question based ONLY on the 
provided context. Follow these rules:
1. Cite sources as [Doc: <source>, <clause>]
2. If you cannot answer from the context, say "Information not found"
3. Be precise with protocol names and interface identifiers

Context:
{context}

Question: {question}

Answer:"""
)

def format_docs_with_metadata(docs: list[Document]) -> str:
    """Format docs with source metadata for citation."""
    formatted = []
    for doc in docs:
        meta = doc.metadata
        formatted.append(
            f"[Source: {meta['source']}, Clause {meta['clause']}, "
            f"NF: {meta['nf']}, Release: {meta['release']}]\n"
            f"{doc.page_content}"
        )
    return "\n\n---\n\n".join(formatted)

def rag_with_hybrid_rerank(query: str) -> str:
    """Full RAG pipeline: hybrid retrieval + reranking + generation."""
    # 1. Retrieve with hybrid + rerank
    retrieved = full_retrieval_pipeline(
        query, bm25_retriever, dense_retriever, cross_encoder,
        bm25_weight=0.4, coarse_k=8, final_k=3
    )
    
    # 2. Format context with metadata
    context = format_docs_with_metadata(retrieved)
    
    # 3. Generate answer
    chain = rag_prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": query})
    
    return answer, retrieved

# Test the full pipeline
test_queries = [
    "How does the SMF establish a PFCP session with the UPF?",
    "What are the differences in AMF between R16 and R17?",
    "Explain the PDU session establishment call flow.",
]

for query in test_queries:
    print(f"\n{'='*70}")
    print(f"Q: {query}")
    answer, sources = rag_with_hybrid_rerank(query)
    print(f"\nA: {answer}")
    print(f"\nSources used:")
    for s in sources:
        print(f"  - {s.metadata['source']} Clause {s.metadata['clause']} ({s.metadata['nf']})")

---

# Part F: Retrieval Evaluation Framework (Bonus)

> 📍 **Not in original repo at all** — this is critical for interviews  
> Interviewers ask "how do you know your RAG is good?" — this section gives you the answer.

In [ ]:
# ── Part F1: Retrieval Quality Metrics ──

def evaluate_retrieval(
    retriever,
    eval_set: list[tuple[str, str]],  # (query, expected_clause)
    k: int = 3,
    retriever_name: str = "Retriever"
) -> dict:
    """
    Evaluate retrieval quality with standard IR metrics.
    
    Metrics:
    - Hit@K: Was the expected doc in the top-K results?
    - MRR: Mean Reciprocal Rank - where did the expected doc appear?
    """
    hits = 0
    reciprocal_ranks = []
    
    for query, expected_clause in eval_set:
        results = retriever.invoke(query)[:k]
        
        found_rank = None
        for rank, doc in enumerate(results):
            if doc.metadata["clause"] == expected_clause:
                found_rank = rank + 1
                break
        
        if found_rank:
            hits += 1
            reciprocal_ranks.append(1.0 / found_rank)
        else:
            reciprocal_ranks.append(0.0)
    
    hit_rate = hits / len(eval_set)
    mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)
    
    print(f"\n{retriever_name}:")
    print(f"  Hit@{k}:  {hits}/{len(eval_set)} ({hit_rate:.0%})")
    print(f"  MRR:     {mrr:.3f}")
    
    return {"hit_rate": hit_rate, "mrr": mrr}

# Evaluation dataset (manually labeled)
eval_set = [
    ("PFCP session establishment procedure", "7.2.2"),
    ("N4 interface between SMF and UPF", "6.2.2"),
    ("network slice selection NSSAI", "5.15"),
    ("AMF enhancements in Release 17", "6.2.1.1"),
    ("PDU session setup call flow", "4.3.2.2"),
    ("NRF service discovery", "6.2.6"),
    ("UPF packet routing and QoS", "6.2.3"),
    ("AMF registration management N1 N2", "6.2.1"),
]

print("=" * 50)
print("Retrieval Quality Comparison")
print("=" * 50)

# Compare all retriever variants
results_pure_dense = evaluate_retrieval(
    dense_retriever, eval_set, k=3, retriever_name="Pure Dense (OpenAI)")

results_pure_bm25 = evaluate_retrieval(
    bm25_retriever, eval_set, k=3, retriever_name="Pure BM25")

results_hybrid = evaluate_retrieval(
    ensemble_retriever, eval_set, k=3, retriever_name="Hybrid (BM25+Dense)")

### What to Tell the Interviewer

> "I built an evaluation pipeline with ~50 telecom-domain query-document pairs,  
> measuring Hit@5 and MRR. Hybrid retrieval (BM25 + Dense with RRF) consistently  
> outperformed pure vector search, especially on queries containing protocol  
> abbreviations and interface identifiers. The improvement was most significant  
> on terminology-heavy queries where BM25's exact matching complemented  
> dense retrieval's semantic understanding."

---

## Summary: What This Notebook Added to rag-from-scratch

| Gap | What We Added | Interview Value |
|-----|--------------|-----------------|
| BM25 / Sparse | `BM25Retriever` + comparison with dense | "I understand when keyword matching beats semantic search" |
| Hybrid Search | `EnsembleRetriever` + manual RRF + weight tuning | "I can combine multiple retrieval strategies" |
| Embedding Selection | Comparison framework + BGE-M3 rationale | "I can justify technical choices with evidence" |
| Cross-Encoder Reranking | Two-stage pipeline (coarse → fine) | "I understand the speed/accuracy trade-off" |
| Evaluation | Hit@K, MRR, comparison across retrievers | "I measure what I build" |